In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path("<PROJECT_ROOT>/mental_health_ft")
RAW_DIR = PROJECT_ROOT / "data" / "raw" / "counsel_chat"

RAW_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw Counsel-Chat directory:", RAW_DIR)

In [ ]:
from huggingface_hub import hf_hub_download

raw_file = hf_hub_download(
    repo_id="nbertagnolli/counsel-chat",
    filename="20220401_counsel_chat.csv",
    repo_type="dataset",
    local_dir=RAW_DIR
)

print("Downloaded to:")
print(raw_file)

In [ ]:
df = pd.read_csv(raw_file)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

In [ ]:
display(df.head(3))

In [ ]:
print("Number of records:", len(df))
print("Number of columns:", len(df.columns))

print("\nMissing values:")
display(df.isna().sum())

print("\nDuplicate rows:", df.duplicated().sum())

In [ ]:
pd.set_option("display.max_colwidth", 300)

display(
    df[
        ["questionID", "questionTitle", "questionText",
         "topic", "answerText", "upvotes"]
    ].head(10)
)

In [ ]:
print("Missing questionText:")
display(
    df[df["questionText"].isna()][
        ["questionID", "questionTitle", "topic", "answerText"]
    ].head(10)
)

print("Missing answerText:")
display(
    df[df["answerText"].isna()][
        ["questionID", "questionTitle", "topic", "questionText"]
    ].head(10)
)

In [ ]:
import re

def normalize_text(text):
    if pd.isna(text):
        return ""
    
    text = str(text).lower()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^\w\s]", "", text)
    return text.strip()

df["question_normalized"] = df["questionText"].apply(normalize_text)

print(
    "Duplicate questions after normalization:",
    df["question_normalized"].duplicated().sum()
)

In [ ]:
duplicates = df[
    df["question_normalized"].duplicated(keep=False)
].sort_values("question_normalized")

display(
    duplicates[
        ["questionID", "questionTitle", "questionText", "answerText"]
    ].head(20)
)

In [ ]:
question_text_status = (
    df.groupby("questionID")["questionText"]
      .agg(
          total_rows="size",
          non_missing=lambda x: x.notna().sum(),
          missing=lambda x: x.isna().sum()
      )
)

print("Total unique question IDs:", len(question_text_status))

print(
    "Question IDs with recoverable missing questionText:",
    (
        (question_text_status["missing"] > 0) &
        (question_text_status["non_missing"] > 0)
    ).sum()
)

print(
    "Question IDs where questionText is missing everywhere:",
    (
        question_text_status["non_missing"] == 0
    ).sum()
)

In [ ]:
missing_question_ids = question_text_status[
    question_text_status["non_missing"] == 0
].index

missing_question_rows = df[
    df["questionID"].isin(missing_question_ids)
]

print("Rows belonging to questionIDs with no questionText:")
print(len(missing_question_rows))

print("\nHow many of these have answerText?")
print(missing_question_rows["answerText"].notna().sum())

print("\nHow many have missing answerText?")
print(missing_question_rows["answerText"].isna().sum())

print("\nNumber of unique questionIDs:")
print(missing_question_rows["questionID"].nunique())

display(
    missing_question_rows[
        ["questionID", "questionTitle", "answerText"]
    ].head(10)
)

In [ ]:
missing_q = missing_question_rows.copy()

missing_q["title_clean"] = (
    missing_q["questionTitle"]
    .fillna("")
    .astype(str)
    .str.strip()
)

print("Total rows:", len(missing_q))

print(
    "Rows with usable-looking questionTitle:",
    (missing_q["title_clean"].str.len() > 10).sum()
)

print(
    "Rows with empty/missing questionTitle:",
    (missing_q["title_clean"].str.len() == 0).sum()
)

print("\nUnique questionIDs with usable-looking titles:")
print(
    missing_q.loc[
        missing_q["title_clean"].str.len() > 10,
        "questionID"
    ].nunique()
)

print("\nShortest titles:")
display(
    missing_q[
        ["questionID", "questionTitle"]
    ]
    .drop_duplicates("questionID")
    .sort_values("questionTitle", key=lambda x: x.str.len())
    .head(20)
)

In [ ]:
cc = df.copy()

# Create the effective user input
cc["user_input"] = cc["questionText"]

# Fallback to questionTitle when questionText is missing
cc["user_input"] = cc["user_input"].fillna(cc["questionTitle"])

# Remove rows where we still don't have a user input
cc_missing_input = cc["user_input"].isna() | (
    cc["user_input"].astype(str).str.strip() == ""
)

# Remove rows without an answer
cc_missing_answer = cc["answerText"].isna() | (
    cc["answerText"].astype(str).str.strip() == ""
)

print("Original rows:", len(cc))
print("Rows missing usable user input:", cc_missing_input.sum())
print("Rows missing usable answer:", cc_missing_answer.sum())

cc_clean_candidate = cc[
    ~cc_missing_input & ~cc_missing_answer
].copy()

print("\nRows remaining:", len(cc_clean_candidate))
print("Unique questionIDs remaining:", cc_clean_candidate["questionID"].nunique())

print("\nInput source:")
print(
    cc_clean_candidate["questionText"].notna()
    .value_counts()
)

print("\nRows recovered from questionTitle:")
print(
    (
        cc_clean_candidate["questionText"].isna()
    ).sum()
)

In [ ]:
# ============================================================
# Check for true duplicate therapist responses
# ============================================================
# 
# We DO NOT want to remove multiple answers just because they
# belong to the same questionID.
#
# One question can have multiple different therapist answers,
# and this diversity is useful for fine-tuning.
#
# Here, we only look for cases where:
#   1. The questionID is the same
#   2. The therapist is the same
#   3. The answer is exactly the same after normalization
#
# These are much stronger candidates for true duplicates.
# ============================================================

import re

def normalize_text(text):
    """
    Normalize text only for duplicate detection.
    
    This does NOT modify the original text.
    It removes differences such as:
    - capitalization
    - extra spaces
    - punctuation
    """
    
    if pd.isna(text):
        return ""
    
    text = str(text).lower()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^\w\s]", "", text)
    
    return text.strip()


# Create normalized versions only for checking duplicates.
cc_clean_candidate["answer_normalized"] = (
    cc_clean_candidate["answerText"]
    .apply(normalize_text)
)

cc_clean_candidate["therapist_normalized"] = (
    cc_clean_candidate["therapistInfo"]
    .apply(normalize_text)
)


# Find rows where the same question + therapist + answer
# appears more than once.
true_duplicate_mask = cc_clean_candidate.duplicated(
    subset=[
        "questionID",
        "therapist_normalized",
        "answer_normalized"
    ],
    keep=False
)

true_duplicates = cc_clean_candidate[
    true_duplicate_mask
].copy()


print("Potential true duplicate rows:", len(true_duplicates))

print(
    "Unique questionIDs containing these duplicates:",
    true_duplicates["questionID"].nunique()
)


# Display a few examples so we can inspect them manually.
display(
    true_duplicates[
        [
            "questionID",
            "questionTitle",
            "therapistInfo",
            "answerText"
        ]
    ].head(20)
)

In [ ]:
# ============================================================
# PII INSPECTION
# ============================================================
#
# We scan the user input and therapist answer for common
# personally identifiable information (PII).
#
# IMPORTANT:
# This cell only DETECTS possible PII.
# It does NOT change or delete any data.
#
# We will inspect the results manually before deciding
# whether any redaction is necessary.
# ============================================================

# Combine the user input and therapist response into one
# text column for inspection.
cc_clean_candidate["combined_text"] = (
    cc_clean_candidate["user_input"].fillna("").astype(str)
    + " "
    + cc_clean_candidate["answerText"].fillna("").astype(str)
)


# ------------------------------------------------------------
# Define common PII patterns
# ------------------------------------------------------------

pii_patterns = {
    # Basic email address pattern
    "email": r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b",

    # International / common phone-number patterns
    # We intentionally keep this conservative because numbers
    # can also represent dates, ages, or other normal text.
    "phone": r"(?<!\d)(?:\+?\d[\d\s().-]{7,}\d)(?!\d)",

    # Web URLs
    "url": r"https?://[^\s]+|www\.[^\s]+",

    # Social-media style handles such as @username
    "social_handle": r"(?<!\w)@[A-Za-z0-9_]{2,30}\b",
}


# ------------------------------------------------------------
# Count possible matches for each PII category
# ------------------------------------------------------------

for pii_type, pattern in pii_patterns.items():

    # Check every row for this type of possible PII.
    matches = cc_clean_candidate["combined_text"].str.contains(
        pattern,
        regex=True,
        case=False,
        na=False
    )

    print(f"{pii_type}: {matches.sum()} rows")


# ------------------------------------------------------------
# Show rows containing possible PII
# ------------------------------------------------------------

# Create one combined mask for any detected PII.
any_pii_mask = pd.Series(
    False,
    index=cc_clean_candidate.index
)

for pattern in pii_patterns.values():

    any_pii_mask |= cc_clean_candidate["combined_text"].str.contains(
        pattern,
        regex=True,
        case=False,
        na=False
    )


possible_pii = cc_clean_candidate[any_pii_mask].copy()

print("\nTotal rows with possible PII:", len(possible_pii))


# Display only the relevant fields for manual inspection.
display(
    possible_pii[
        [
            "questionID",
            "user_input",
            "answerText"
        ]
    ].head(20)
)

In [ ]:
# ============================================================
# PII FLAG BREAKDOWN
# ============================================================
#
# The previous scan told us that 172 rows contain something
# that LOOKS like PII.
#
# We should not delete or redact them automatically because:
#   - URLs may simply be educational resources.
#   - Numbers may be ages, dates, etc.
#   - @ symbols may not be personal social-media handles.
#
# This cell identifies exactly which pattern triggered each row.
# ============================================================


# Create a table to store which PII patterns were detected.
pii_check = pd.DataFrame(index=cc_clean_candidate.index)


# Run every PII detector separately.
for pii_type, pattern in pii_patterns.items():

    pii_check[pii_type] = cc_clean_candidate[
        "combined_text"
    ].str.contains(
        pattern,
        regex=True,
        case=False,
        na=False
    )


# Count how many rows triggered each detector.
print("Rows flagged by each detector:\n")

for pii_type in pii_patterns:
    print(
        f"{pii_type}: "
        f"{pii_check[pii_type].sum()} rows"
    )


# Count how many different detectors triggered for each row.
pii_check["number_of_flags"] = pii_check.sum(axis=1)


print("\nNumber of PII flags per row:")
print(
    pii_check["number_of_flags"]
    .value_counts()
    .sort_index()
)


# Add the number of flags to the candidate dataframe.
cc_clean_candidate["pii_flags"] = pii_check["number_of_flags"]


# Show rows that were flagged by the phone detector.
print("\nExamples flagged as possible phone numbers:")

phone_rows = cc_clean_candidate[
    pii_check["phone"]
]

display(
    phone_rows[
        [
            "questionID",
            "user_input",
            "answerText"
        ]
    ].head(15)
)

In [ ]:
# ============================================================
# INSPECT POSSIBLE SOCIAL-MEDIA HANDLES
# ============================================================
#
# Only 3 rows were flagged by the @username pattern.
#
# The @ symbol can appear for many reasons, so we inspect
# these rows manually before deciding whether they contain
# actual personal information.
# ============================================================

social_rows = cc_clean_candidate[
    pii_check["social_handle"]
].copy()

print("Rows flagged for possible social handles:", len(social_rows))

display(
    social_rows[
        [
            "questionID",
            "user_input",
            "answerText"
        ]
    ]
)

In [ ]:
# ============================================================
# Check for HTML / formatting artifacts
# ============================================================

# We only inspect the therapist answers.
answer_text = (
    cc_clean_candidate["answerText"]
    .fillna("")
    .astype(str)
)

# Look for HTML tags such as <br>, <p>, <div>, etc.
html_tag_pattern = r"<[^>]+>"

# Look for HTML comments such as <!-- ... -->
html_comment_pattern = r"<!--.*?-->"

# Check each answer for HTML tags.
has_html_tags = answer_text.str.contains(
    html_tag_pattern,
    regex=True,
    na=False
)

# Check each answer for HTML comments.
has_html_comments = answer_text.str.contains(
    html_comment_pattern,
    regex=True,
    na=False
)

# Combine both checks.
has_html = has_html_tags | has_html_comments

print("Answers containing HTML tags:", has_html_tags.sum())
print("Answers containing HTML comments:", has_html_comments.sum())
print("Answers containing any HTML artifact:", has_html.sum())

In [ ]:
# ============================================================
# Inspect the exact text detected as possible phone numbers
# ============================================================

import re

# Use the same phone pattern from our previous check.
phone_pattern = pii_patterns["phone"]

# Find the actual matching text in each answer/user input.
def find_phone_matches(text):
    """
    Return the exact strings detected by the phone regex.
    This is for inspection only.
    """
    
    if pd.isna(text):
        return []
    
    return re.findall(
        phone_pattern,
        str(text)
    )


# Apply the detector to the complete conversational text.
cc_clean_candidate["phone_matches"] = (
    cc_clean_candidate["combined_text"]
    .apply(find_phone_matches)
)

# Keep only rows where something was actually detected.
phone_matches_df = cc_clean_candidate[
    cc_clean_candidate["phone_matches"].str.len() > 0
].copy()

print("Rows with possible phone matches:", len(phone_matches_df))

# Show the exact detected strings.
display(
    phone_matches_df[
        [
            "questionID",
            "phone_matches",
            "user_input",
            "answerText"
        ]
    ]
)

In [ ]:
# ============================================================
# Counsel-Chat: Phone Number Cleaning
# ============================================================
#
# We work on a COPY of the dataset.
# The original raw dataset will remain unchanged.
# ============================================================

import re
import pandas as pd

# Make a copy so that the raw/previous dataset is preserved.
cc_phone_clean = cc_clean_candidate.copy()


# ------------------------------------------------------------
# Phone-number pattern
# ------------------------------------------------------------
#
# This pattern is the same detector we used during inspection.
# It finds number sequences that look like phone numbers.
#
# IMPORTANT:
# We will inspect the matches before deciding how to replace
# them.
# ------------------------------------------------------------

phone_pattern = pii_patterns["phone"]


# ------------------------------------------------------------
# Function to find phone-number matches
# ------------------------------------------------------------

def find_phone_matches(text):
    """
    Return all possible phone-number matches in a text.
    """
    
    if pd.isna(text):
        return []
    
    return re.findall(
        phone_pattern,
        str(text)
    )


# ------------------------------------------------------------
# Check the combined conversation text
# ------------------------------------------------------------

cc_phone_clean["phone_matches"] = (
    cc_phone_clean["combined_text"]
    .apply(find_phone_matches)
)


# Keep only rows where a possible phone number was detected.
phone_rows = cc_phone_clean[
    cc_phone_clean["phone_matches"].str.len() > 0
].copy()


print("Rows with possible phone numbers:", len(phone_rows))


# Show the exact detected values.
display(
    phone_rows[
        [
            "questionID",
            "phone_matches"
        ]
    ]
)

In [ ]:
# ============================================================
# COUNSEL-CHAT CLEANING PIPELINE
# Cell 1 — Create a working copy
# ============================================================

import re
import html
import pandas as pd
import numpy as np

# IMPORTANT:
# Never modify the raw Counsel-Chat dataframe directly.
# We create a separate working copy.
cc_clean = cc_clean_candidate.copy()

print("Starting rows:", len(cc_clean))
print("Starting unique questionIDs:", cc_clean["questionID"].nunique())

In [ ]:
# ============================================================
# Cell 2 — Prepare user input and therapist answer
# ============================================================

# user_input should already contain:
# questionText when available, otherwise questionTitle.
#
# answerText contains the therapist's response.

cc_clean["user_input"] = (
    cc_clean["user_input"]
    .fillna("")
    .astype(str)
    .str.strip()
)

cc_clean["answerText"] = (
    cc_clean["answerText"]
    .fillna("")
    .astype(str)
    .str.strip()
)

# Remove accidental "nan" strings if they were created by astype(str).
cc_clean["user_input"] = cc_clean["user_input"].replace("nan", "")
cc_clean["answerText"] = cc_clean["answerText"].replace("nan", "")

print("Missing user input:", (cc_clean["user_input"] == "").sum())
print("Missing answers:", (cc_clean["answerText"] == "").sum())

In [ ]:
# ============================================================
# Cell 3 — Remove HTML / formatting artifacts
# ============================================================

def clean_html_artifacts(text):
    """
    Remove HTML comments, HTML tags, and common CSS artifacts
    while preserving normal conversational text.
    """
    
    if pd.isna(text):
        return ""
    
    text = str(text)
    
    # Decode HTML entities such as &amp; or &nbsp;.
    text = html.unescape(text)
    
    # Remove HTML comments:
    # <!-- ... -->
    text = re.sub(
        r"<!--.*?-->",
        " ",
        text,
        flags=re.DOTALL
    )
    
    # Remove HTML tags such as:
    # <br>, <p>, <div>, etc.
    text = re.sub(
        r"<[^>]+>",
        " ",
        text
    )
    
    # Remove obvious CSS @font-face blocks.
    text = re.sub(
        r"@font-face.*?(?=\n\s*\n|\Z)",
        " ",
        text,
        flags=re.DOTALL | re.IGNORECASE
    )
    
    # Remove common CSS/font metadata fragments.
    text = re.sub(
        r"mso-[\w-]+\s*:[^;{}]+;?",
        " ",
        text,
        flags=re.IGNORECASE
    )
    
    return text


# Apply only to text fields used for training.
cc_clean["user_input"] = cc_clean["user_input"].apply(clean_html_artifacts)
cc_clean["answerText"] = cc_clean["answerText"].apply(clean_html_artifacts)

print("HTML cleaning completed.")

In [ ]:
c

In [ ]:
# ============================================================
# Cell 4 — Remove / anonymize phone numbers
# ============================================================

# We previously inspected the phone-number detector results.
#
# Important:
# Some detected "phone numbers" were NOT phone numbers.
# For example, CSS/font metadata contained long numeric sequences
# such as:
#     536870145 1791491579 18 0 131231 0
#
# Therefore, we use a stricter pattern for normal phone formats.
#
# We also distinguish between:
#   [CRISIS_HOTLINE]  -> known mental-health/crisis hotline numbers
#   [PHONE_NUMBER]    -> other genuine phone numbers
#
# We do NOT delete the surrounding answer.
# We only replace the sensitive number with a meaningful tag.


# ------------------------------------------------------------
# 1. Known hotline numbers found in Counsel-Chat
# ------------------------------------------------------------

# These numbers appeared in the dataset during our inspection.
# We preserve the fact that they are crisis/support resources
# without keeping the actual phone number.

CRISIS_HOTLINES = {
    "1-800-273-8255",
    "800-273-8255",
    "1-800-422-4453",
    "888-724-7240",
    "800-989-6884",
    "787-466-5478",
    "1-800-246-7743",
    "1-877-656-8860",
    "1-866-488-7386",
    "877-565-8860",
    "877-330-6366",
}


# ------------------------------------------------------------
# 2. Phone-number pattern
# ------------------------------------------------------------

# We only target recognizable phone-number structures.
#
# Examples:
#   1-800-273-8255
#   800-273-8255
#   (514) 690-2469
#   404.293.5654
#   305-374-9990
#   787-466-5478
#
# We intentionally DO NOT match arbitrary sequences of numbers.

phone_pattern = re.compile(
    r"""
    (?<!\d)
    (?:
        # 1-800-123-4567
        1[-.\s]\d{3}[-.\s]\d{3}[-.\s]\d{4}

        |

        # 800-123-4567
        \d{3}[-.\s]\d{3}[-.\s]\d{4}

        |

        # (514) 690-2469
        \(\d{3}\)[-\s.]?\d{3}[-.\s]\d{4}

        |

        # 514-690-2469 / 514.690.2469
        \d{3}[-.]\d{3}[-.]\d{4}

    )
    (?!\d)
    """,
    re.VERBOSE
)


# ------------------------------------------------------------
# 3. Normalize a phone number for comparison
# ------------------------------------------------------------

def normalize_phone(phone):
    """
    Remove formatting characters so that different representations
    of the same phone number can be compared.

    Example:
        1-800-273-8255 -> 18002738255
        800-273-8255   -> 8002738255
    """
    
    return re.sub(r"\D", "", phone)


# Create normalized versions of the known crisis numbers.
normalized_crisis_hotlines = {
    normalize_phone(phone)
    for phone in CRISIS_HOTLINES
}


# ------------------------------------------------------------
# 4. Replace detected phone numbers
# ------------------------------------------------------------

def anonymize_phone_numbers(text):
    """
    Replace genuine phone numbers with meaningful tags.

    Known crisis/support numbers:
        [CRISIS_HOTLINE]

    Other phone numbers:
        [PHONE_NUMBER]

    Everything else remains unchanged.
    """

    if pd.isna(text):
        return ""

    text = str(text)

    def replace_match(match):

        original_number = match.group(0)

        normalized = normalize_phone(original_number)

        # Check whether this is one of the known crisis/support
        # numbers found during our dataset inspection.
        if normalized in normalized_crisis_hotlines:
            return "[CRISIS_HOTLINE]"

        # Otherwise, anonymize it as a general phone number.
        return "[PHONE_NUMBER]"

    return phone_pattern.sub(
        replace_match,
        text
    )


# ------------------------------------------------------------
# 5. Apply to user input
# ------------------------------------------------------------

cc_clean["user_input"] = cc_clean["user_input"].apply(
    anonymize_phone_numbers
)


# ------------------------------------------------------------
# 6. Apply to therapist answers
# ------------------------------------------------------------

cc_clean["answerText"] = cc_clean["answerText"].apply(
    anonymize_phone_numbers
)


# ------------------------------------------------------------
# 7. Verification
# ------------------------------------------------------------

print("Phone-number anonymization completed.")

print(
    "Remaining possible phone numbers:",
    cc_clean["user_input"].str.contains(
        phone_pattern,
        regex=True,
        na=False
    ).sum()
    +
    cc_clean["answerText"].str.contains(
        phone_pattern,
        regex=True,
        na=False
    ).sum()
)

print(
    "CRISIS_HOTLINE tags:",
    (
        cc_clean["user_input"].str.contains(
            r"\[CRISIS_HOTLINE\]",
            regex=True,
            na=False
        )
        |
        cc_clean["answerText"].str.contains(
            r"\[CRISIS_HOTLINE\]",
            regex=True,
            na=False
        )
    ).sum()
)

print(
    "PHONE_NUMBER tags:",
    (
        cc_clean["user_input"].str.contains(
            r"\[PHONE_NUMBER\]",
            regex=True,
            na=False
        )
        |
        cc_clean["answerText"].str.contains(
            r"\[PHONE_NUMBER\]",
            regex=True,
            na=False
        )
    ).sum()
)

In [ ]:
# ============================================================
# Cell 6 — Remove obvious social-media handles
# ============================================================

def anonymize_social_handles(text):
    """
    Replace obvious @username patterns.

    We do NOT remove every '@' character because it can occur
    inside emails or technical/formatting text.
    """
    
    if pd.isna(text):
        return ""
    
    text = str(text)
    
    # @username pattern
    text = re.sub(
        r"(?<![\w@])@[A-Za-z0-9_]{2,30}",
        "[SOCIAL_HANDLE]",
        text
    )
    
    return text


cc_clean["user_input"] = cc_clean["user_input"].apply(
    anonymize_social_handles
)

cc_clean["answerText"] = cc_clean["answerText"].apply(
    anonymize_social_handles
)

print("Social-handle cleaning completed.")

In [ ]:
# ============================================================
# Cell 7 — Normalize whitespace
# ============================================================

def normalize_text(text):
    """
    Normalize whitespace without aggressively rewriting the
    therapist's language.
    """
    
    if pd.isna(text):
        return ""
    
    text = str(text)
    
    # Normalize non-breaking spaces.
    text = text.replace("\xa0", " ")
    
    # Normalize line breaks / tabs / repeated spaces.
    text = re.sub(
        r"\s+",
        " ",
        text
    )
    
    return text.strip()


cc_clean["user_input"] = cc_clean["user_input"].apply(
    normalize_text
)

cc_clean["answerText"] = cc_clean["answerText"].apply(
    normalize_text
)

print("Whitespace normalization completed.")

In [ ]:
# ============================================================
# Cell 8 — Check for empty training fields
# ============================================================

empty_user = cc_clean["user_input"].eq("")
empty_answer = cc_clean["answerText"].eq("")

print("Rows with empty user input:", empty_user.sum())
print("Rows with empty answer:", empty_answer.sum())

# Remove only rows that cannot form a training example.
cc_clean = cc_clean[
    ~empty_user & ~empty_answer
].copy()

print("\nRows after removing unusable examples:", len(cc_clean))
print(
    "Unique questionIDs:",
    cc_clean["questionID"].nunique()
)

In [ ]:
# ============================================================
# Cell 9 — Final phone-number verification
# ============================================================

# Search for common phone-number structures again.
remaining_phone_matches = (
    cc_clean["user_input"].str.findall(phone_clean_pattern)
    .str.len()
    +
    cc_clean["answerText"].str.findall(phone_clean_pattern)
    .str.len()
)

print(
    "Rows with remaining phone-number patterns:",
    (remaining_phone_matches > 0).sum()
)

In [ ]:
# ============================================================
# Cell 10 — Final URL verification
# ============================================================

remaining_urls = (
    cc_clean["user_input"].str.contains(
        url_pattern,
        regex=True,
        na=False
    )
    |
    cc_clean["answerText"].str.contains(
        url_pattern,
        regex=True,
        na=False
    )
)

print(
    "Rows with remaining URLs:",
    remaining_urls.sum()
)

In [ ]:
# ============================================================
# Cell 11 — Final HTML verification
# ============================================================

html_check_pattern = r"<[^>]+>"

remaining_html = (
    cc_clean["user_input"].str.contains(
        html_check_pattern,
        regex=True,
        na=False
    )
    |
    cc_clean["answerText"].str.contains(
        html_check_pattern,
        regex=True,
        na=False
    )
)

print(
    "Rows with remaining HTML tags:",
    remaining_html.sum()
)

In [ ]:
# ============================================================
# Cell 12 — Final duplicate check
# ============================================================

# Exact duplicate training pairs.
duplicate_pairs = cc_clean.duplicated(
    subset=["user_input", "answerText"],
    keep=False
)

print(
    "Potential duplicate training pairs:",
    duplicate_pairs.sum()
)

print(
    "Unique questionIDs containing duplicate pairs:",
    cc_clean.loc[
        duplicate_pairs,
        "questionID"
    ].nunique()
)

In [ ]:
# ============================================================
# Cell 13 — Final Counsel-Chat cleaning summary
# ============================================================

print("=" * 60)
print("FINAL COUNSEL-CHAT CLEANING SUMMARY")
print("=" * 60)

print("Rows:", len(cc_clean))
print("Unique questionIDs:", cc_clean["questionID"].nunique())

print(
    "Missing user input:",
    cc_clean["user_input"].eq("").sum()
)

print(
    "Missing answer:",
    cc_clean["answerText"].eq("").sum()
)

print(
    "Remaining phone patterns:",
    (remaining_phone_matches > 0).sum()
)

print(
    "Remaining URLs:",
    remaining_urls.sum()
)

print(
    "Remaining HTML:",
    remaining_html.sum()
)

print(
    "Exact duplicate training pairs:",
    duplicate_pairs.sum()
)

In [ ]:
# ============================================================
# Cell 14 — Inspect cleaned examples
# ============================================================

# Show rows that contain our replacement tokens.
cleaned_examples = cc_clean[
    cc_clean["user_input"].str.contains(
        r"\[PHONE\]|\[URL\]|\[SOCIAL_HANDLE\]",
        regex=True,
        na=False
    )
    |
    cc_clean["answerText"].str.contains(
        r"\[PHONE\]|\[URL\]|\[SOCIAL_HANDLE\]",
        regex=True,
        na=False
    )
]

print(
    "Rows containing anonymization tokens:",
    len(cleaned_examples)
)

display(
    cleaned_examples[
        [
            "questionID",
            "user_input",
            "answerText"
        ]
    ].head(20)
)

In [ ]:
# ============================================================
# Final QA — Duplicate checks
# ============================================================

print("\n--- Duplicate checks ---")

# IMPORTANT:
# cc_clean may contain detector/QA columns whose values are lists
# (for example phone_matches). Pandas cannot use list values when
# calculating duplicated() across the entire DataFrame.
#
# Therefore, we check duplicates only on the actual dataset fields
# that will be used for training.

# ------------------------------------------------------------
# 1. Exact duplicate training examples
# ------------------------------------------------------------

duplicate_rows = cc_clean.duplicated(
    subset=["questionID", "user_input", "answerText"],
    keep=False
).sum()

print(f"Exact duplicate training rows: {duplicate_rows}")


# ------------------------------------------------------------
# 2. Duplicate questionID + answerText
# ------------------------------------------------------------
# Multiple answers for the same questionID are EXPECTED in
# Counsel-Chat because different therapists can answer the
# same question.
#
# Therefore, we do NOT remove these as duplicates.

duplicate_id_answer = cc_clean.duplicated(
    subset=["questionID", "answerText"],
    keep=False
).sum()

print(f"Duplicate questionID + answerText pairs: {duplicate_id_answer}")


# ------------------------------------------------------------
# 3. Duplicate questionID + user_input
# ------------------------------------------------------------
# This is also expected because one user question can have
# multiple therapist answers.

duplicate_id_input = cc_clean.duplicated(
    subset=["questionID", "user_input"],
    keep=False
).sum()

print(f"Rows sharing questionID + user_input: {duplicate_id_input}")


# ------------------------------------------------------------
# 4. Number of unique question IDs
# ------------------------------------------------------------

unique_question_ids = cc_clean["questionID"].nunique()

print(f"Unique questionIDs: {unique_question_ids}")


# ------------------------------------------------------------
# 5. Final expected relationship
# ------------------------------------------------------------

print(
    f"Rows: {len(cc_clean):,} | "
    f"Unique questionIDs: {unique_question_ids:,}"
)

In [ ]:
import re
import json
from pathlib import Path
from sklearn.model_selection import GroupShuffleSplit

SEED = 42
OUT_DIR = Path("<PROJECT_ROOT>/mental_health_ft")
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
print("Rows before final preprocessing:", len(cc_clean))
print("Unique questionIDs:", cc_clean["questionID"].nunique())

In [ ]:
url_pattern = re.compile(r"https?://\S+|www.\S+", re.IGNORECASE)

def anonymize_urls(text):
    
    if pd.isna(text):
       return ""
    return url_pattern.sub("[URL]", str(text))

In [ ]:
cc_clean["user_input"] = cc_clean["user_input"].apply(anonymize_urls)
cc_clean["answerText"] = cc_clean["answerText"].apply(anonymize_urls)

In [ ]:
remaining_urls = (
cc_clean["user_input"].str.contains(url_pattern, regex=True, na=False)
| cc_clean["answerText"].str.contains(url_pattern, regex=True, na=False)
)
print("Rows with remaining URL-like strings:", remaining_urls.sum())

In [ ]:

# ------------------ Cell C — final PII / phone / HTML checks ------------------

import re
import pandas as pd

# Sanity checks: ensure expected variables exist
if "cc_clean" not in globals():
    raise RuntimeError("cc_clean not found. Run earlier cells to create it.")

if "OUT_DIR" not in globals():
    from pathlib import Path

    OUT_DIR = Path("<PROJECT_ROOT>/mental_health_ft")
    OUT_DIR.mkdir(parents=True, exist_ok=True)

if "SEED" not in globals():
    SEED = 42


# Conservative phone pattern (same as used earlier)
phone_pattern = re.compile(
    r"""
    (?<!\d)
    (?:
        1[-.\s]\d{3}[-.\s]\d{3}[-.\s]\d{4}      # 1-800-123-4567
        |
        \d{3}[-.\s]\d{3}[-.\s]\d{4}             # 800-123-4567
        |
        \(\d{3}\)[-\s.]?\d{3}[-.\s]\d{4}        # (514) 690-2469
        |
        \d{3}[-.]\d{3}[-.]\d{4}                 # 514-690-2469
    )
    (?!\d)
    """,
    re.VERBOSE,
)


# HTML tag checker
html_check_pattern = re.compile(r"<[^>]+>")


# Count remaining phone-like matches
num_phone = (
    cc_clean["user_input"]
    .str.contains(phone_pattern, regex=True, na=False)
    .sum()
    +
    cc_clean["answerText"]
    .str.contains(phone_pattern, regex=True, na=False)
    .sum()
)


# Count remaining HTML tags
num_html = (
    cc_clean["user_input"]
    .str.contains(html_check_pattern, regex=True, na=False)
    .sum()
    +
    cc_clean["answerText"]
    .str.contains(html_check_pattern, regex=True, na=False)
    .sum()
)


print("Remaining phone-like matches (should be 0):", num_phone)
print("Remaining HTML tags (should be 0):", num_html)


# Basic emptiness checks
print(
    "Rows with empty user_input:",
    cc_clean["user_input"].eq("").sum()
)

print(
    "Rows with empty answerText:",
    cc_clean["answerText"].eq("").sum()
)



In [ ]:

# ------------------ Cell D — mark / export crisis-related rows for manual review ------------------

import re

# Conservative crisis keyword list
crisis_keywords = [
    r"\bkill\b",
    r"\bsuicid",                    # suicide, suicidal, suicidality, etc.
    r"\bi want to die\b",
    r"\bharm myself\b",
    r"\bthoughts of killing\b",
    r"\bwant to die\b",
    r"\bending my life\b",
]


# Compile crisis pattern
crisis_re = re.compile(
    "|".join(crisis_keywords),
    re.IGNORECASE
)


def is_crisis_row(row):
    text = f"{row['user_input']} {row['answerText']}"

    return bool(
        crisis_re.search(text)
        or "[CRISIS_HOTLINE]" in text
    )


# Create crisis candidate flag
cc_clean["is_crisis_candidate"] = cc_clean.apply(
    is_crisis_row,
    axis=1
)


# Extract crisis candidates for manual review
crisis_df = cc_clean[
    cc_clean["is_crisis_candidate"]
].copy()


print("Crisis candidate rows:", len(crisis_df))



In [ ]:

# ------------------ Cell E — group-split by questionID into train / val / test (90/5/5) ------------------

from sklearn.model_selection import GroupShuffleSplit

# Group split to avoid prompt/question leakage.
# All rows belonging to the same questionID must stay in the same split.

groups = cc_clean["questionID"].values


# First split: 90% train / 10% temporary
gss = GroupShuffleSplit(
    n_splits=1,
    train_size=0.90,
    random_state=SEED
)

train_idx, temp_idx = next(
    gss.split(cc_clean, groups=groups)
)

train_df = (
    cc_clean.iloc[train_idx]
    .copy()
    .reset_index(drop=True)
)

temp = (
    cc_clean.iloc[temp_idx]
    .copy()
    .reset_index(drop=True)
)


# Second split: divide the remaining 10% equally
# -> approximately 5% validation / 5% test

gss2 = GroupShuffleSplit(
    n_splits=1,
    train_size=0.50,
    random_state=SEED
)

val_idx_rel, test_idx_rel = next(
    gss2.split(temp, groups=temp["questionID"].values)
)

val_df = (
    temp.iloc[val_idx_rel]
    .copy()
    .reset_index(drop=True)
)

test_df = (
    temp.iloc[test_idx_rel]
    .copy()
    .reset_index(drop=True)
)


# Display split statistics
print("TRAIN")
print("  Rows:", len(train_df))
print("  Unique questionIDs:", train_df["questionID"].nunique())

print("\nVALIDATION")
print("  Rows:", len(val_df))
print("  Unique questionIDs:", val_df["questionID"].nunique())

print("\nTEST")
print("  Rows:", len(test_df))
print("  Unique questionIDs:", test_df["questionID"].nunique())


# ------------------------------------------------------------------
# Sanity check: no questionID may appear in more than one split
# ------------------------------------------------------------------

train_qids = set(train_df["questionID"])
val_qids = set(val_df["questionID"])
test_qids = set(test_df["questionID"])

intersect_train_val = train_qids.intersection(val_qids)
intersect_train_test = train_qids.intersection(test_qids)
intersect_val_test = val_qids.intersection(test_qids)

assert not intersect_train_val, "Group leakage detected between TRAIN and VAL!"
assert not intersect_train_test, "Group leakage detected between TRAIN and TEST!"
assert not intersect_val_test, "Group leakage detected between VAL and TEST!"

print("\nGroup leakage check: PASSED")
print("No questionID appears in more than one split.")



In [ ]:

# ------------------ Cell I — export JSONL and Hugging Face datasets ------------------

import json
from datasets import Dataset


# ------------------------------------------------------------------
# Helper: save DataFrame as JSONL
# ------------------------------------------------------------------

def df_to_jsonl(df, path):
    """
    Save a DataFrame as UTF-8 JSON Lines.
    Each DataFrame row is written as one JSON object.
    """

    with open(path, "w", encoding="utf-8") as f:
        for row in df.to_dict(orient="records"):
            f.write(
                json.dumps(
                    row,
                    ensure_ascii=False
                )
                + "\n"
            )


# ------------------------------------------------------------------
# Save JSONL files
# ------------------------------------------------------------------

train_jsonl = OUT_DIR / "counsel_chat_train.jsonl"
val_jsonl = OUT_DIR / "counsel_chat_val.jsonl"
test_jsonl = OUT_DIR / "counsel_chat_test.jsonl"

df_to_jsonl(train_df, train_jsonl)
df_to_jsonl(val_df, val_jsonl)
df_to_jsonl(test_df, test_jsonl)


# ------------------------------------------------------------------
# Save Hugging Face Dataset format
# ------------------------------------------------------------------

train_hf_path = OUT_DIR / "hf_counsel_train"
val_hf_path = OUT_DIR / "hf_counsel_val"
test_hf_path = OUT_DIR / "hf_counsel_test"


Dataset.from_pandas(
    train_df,
    preserve_index=False
).save_to_disk(train_hf_path)


Dataset.from_pandas(
    val_df,
    preserve_index=False
).save_to_disk(val_hf_path)


Dataset.from_pandas(
    test_df,
    preserve_index=False
).save_to_disk(test_hf_path)


# ------------------------------------------------------------------
# Verify saved outputs
# ------------------------------------------------------------------

print("JSONL files saved:")
print("  ✓", train_jsonl)
print("  ✓", val_jsonl)
print("  ✓", test_jsonl)

print("\nHugging Face datasets saved:")
print("  ✓", train_hf_path)
print("  ✓", val_hf_path)
print("  ✓", test_hf_path)


# ------------------------------------------------------------------
# Quick verification
# ------------------------------------------------------------------

train_hf = Dataset.load_from_disk(train_hf_path)
val_hf = Dataset.load_from_disk(val_hf_path)
test_hf = Dataset.load_from_disk(test_hf_path)

print("\nVerification:")
print("  Train:", len(train_hf), "rows")
print("  Val:  ", len(val_hf), "rows")
print("  Test: ", len(test_hf), "rows")

assert len(train_hf) == len(train_df)
assert len(val_hf) == len(val_df)
assert len(test_hf) == len(test_df)

print("\nExport verification: PASSED")


In [ ]:

# ------------------ Save & verify processed splits ------------------

from pathlib import Path
import json
import pandas as pd


# ================================================================
# 1. Ensure output directory exists
# ================================================================

OUT_DIR = Path(
    "<PROJECT_ROOT>/mental_health_ft"
)

OUT_DIR.mkdir(parents=True, exist_ok=True)


# ================================================================
# 2. Check required split objects exist
# ================================================================

for name in ["train_df", "val_df", "test_df"]:
    if name not in globals():
        raise RuntimeError(
            f"{name} not found in environment. "
            "Run the split cell first."
        )

train_df = globals()["train_df"]
val_df = globals()["val_df"]
test_df = globals()["test_df"]


# ================================================================
# 3. Save CSV files
# ================================================================

train_csv = OUT_DIR / "counsel_chat_train.csv"
val_csv = OUT_DIR / "counsel_chat_val.csv"
test_csv = OUT_DIR / "counsel_chat_test.csv"

train_df.to_csv(
    train_csv,
    index=False,
    encoding="utf-8-sig"
)

val_df.to_csv(
    val_csv,
    index=False,
    encoding="utf-8-sig"
)

test_df.to_csv(
    test_csv,
    index=False,
    encoding="utf-8-sig"
)


# ================================================================
# 4. Save JSONL files
# ================================================================

def df_to_jsonl(df, path):
    """Save a DataFrame as UTF-8 JSONL."""

    with open(path, "w", encoding="utf-8") as f:
        for row in df.to_dict(orient="records"):
            f.write(
                json.dumps(
                    row,
                    ensure_ascii=False
                )
                + "\n"
            )


train_jsonl = OUT_DIR / "counsel_chat_train.jsonl"
val_jsonl = OUT_DIR / "counsel_chat_val.jsonl"
test_jsonl = OUT_DIR / "counsel_chat_test.jsonl"

df_to_jsonl(train_df, train_jsonl)
df_to_jsonl(val_df, val_jsonl)
df_to_jsonl(test_df, test_jsonl)


# ================================================================
# 5. Save Hugging Face datasets
# ================================================================

hf_saved = False

try:
    from datasets import Dataset

    hf_train_dir = OUT_DIR / "hf_counsel_train"
    hf_val_dir = OUT_DIR / "hf_counsel_val"
    hf_test_dir = OUT_DIR / "hf_counsel_test"

    Dataset.from_pandas(
        train_df,
        preserve_index=False
    ).save_to_disk(hf_train_dir)

    Dataset.from_pandas(
        val_df,
        preserve_index=False
    ).save_to_disk(hf_val_dir)

    Dataset.from_pandas(
        test_df,
        preserve_index=False
    ).save_to_disk(hf_test_dir)

    hf_saved = True

except Exception as e:
    print(
        "Hugging Face save_to_disk skipped "
        "(datasets not available or save failed):",
        e
    )


# ================================================================
# 6. Save crisis candidates if available
# ================================================================

crisis_csv = None

if "crisis_df" in globals():

    crisis_csv = (
        OUT_DIR /
        "counsel_chat_crisis_candidates.csv"
    )

    crisis_df.to_csv(
        crisis_csv,
        index=False,
        encoding="utf-8-sig"
    )


# ================================================================
# 7. Verification summary
# ================================================================

print("\n" + "=" * 60)
print("SAVED FILES")
print("=" * 60)

print("Output directory:")
print(OUT_DIR)

print("\nFiles:")

for p in sorted(OUT_DIR.iterdir()):
    if p.is_file():
        print(
            f"- {p.name} ({p.stat().st_size:,} bytes)"
        )
    elif p.is_dir():
        print(
            f"- {p.name}/ (Hugging Face dataset)"
        )


# ================================================================
# 8. Summary counts
# ================================================================

print("\n" + "=" * 60)
print("SUMMARY COUNTS")
print("=" * 60)

print("TRAIN rows:", len(train_df))
print("VAL rows:  ", len(val_df))
print("TEST rows: ", len(test_df))

print(
    "TRAIN unique questionIDs:",
    train_df["questionID"].nunique()
)

print(
    "VAL unique questionIDs:",
    val_df["questionID"].nunique()
)

print(
    "TEST unique questionIDs:",
    test_df["questionID"].nunique()
)

if crisis_csv is not None:
    print(
        "Crisis candidates saved:",
        crisis_csv.name
    )
else:
    print(
        "Crisis candidates saved: no "
        "(crisis_df was not present)"
    )

print(
    "HF datasets saved:",
    "yes" if hf_saved else "no"
)


# ================================================================
# 9. Final verification
# ================================================================

assert train_csv.exists()
assert val_csv.exists()
assert test_csv.exists()

assert train_jsonl.exists()
assert val_jsonl.exists()
assert test_jsonl.exists()

if hf_saved:
    assert hf_train_dir.exists()
    assert hf_val_dir.exists()
    assert hf_test_dir.exists()

print("\nExport verification: PASSED")
print("All required processed splits were saved successfully.")


In [ ]:
%pip install -q datasets